# 05: Offline Evaluation

This notebook walks through the **Phase 2 offline evaluation suite** for the
Knowledge-Grounded QA Agent. Each evaluator runs *after* the agent has responded,
using a Langfuse dataset as the source of truth.

## What You'll Learn

1. How to run the agent and capture the full `AgentResponse` (plan, tool calls, sources)
2. **Replanning Rate** — deterministic counter (no LLM needed)
3. **Plan Quality** — LLM judge on the research plan
4. **Tool Selection & Efficiency** — LLM judge on the tool call sequence
5. **Source Validation** — LLM judge on source authority and relevance
6. **Knowledge Base Usage** — checks vertex_search was called when required
7. Running the full experiment with `run_experiment` and all evaluators wired in

## Prerequisites

Complete Notebooks 01–03. All credentials in `.env`:
- `GOOGLE_API_KEY`
- `LANGFUSE_PUBLIC_KEY` and `LANGFUSE_SECRET_KEY`
- `OPENAI_API_KEY` (for LLM judges)
- `VERTEX_AI_DATASTORE_ID` (optional — only for KB evaluation)

In [ ]:
import os
from pathlib import Path
from typing import Any

import pandas as pd
from aieng.agent_evals.evaluation import run_experiment
from aieng.agent_evals.knowledge_qa import DeepSearchQADataset, KnowledgeGroundedAgent
from aieng.agent_evals.knowledge_qa.evaluation.graders.plan_quality import (
    derive_plan_rubric,
    evaluate_plan_quality,
)
from aieng.agent_evals.knowledge_qa.evaluation.graders.replanning import (
    evaluate_replanning_rate,
    get_max_replan_threshold,
)
from aieng.agent_evals.knowledge_qa.evaluation.graders.source_validation import (
    evaluate_source_validation,
    get_source_rubric,
)
from aieng.agent_evals.knowledge_qa.evaluation.graders.tool_selection import (
    derive_tool_pattern,
    evaluate_tool_selection,
)
from aieng.agent_evals.knowledge_qa.evaluation.offline import (
    KnowledgeQATask,
    deepsearchqa_evaluator,
    knowledge_base_evaluator,
    plan_quality_evaluator,
    replanning_evaluator,
    source_validation_evaluator,
    tool_selection_evaluator,
)
from aieng.agent_evals.knowledge_qa.notebook import display_response, run_with_display
from dotenv import load_dotenv
from IPython.display import HTML, display  # noqa: A004
from rich.console import Console
from rich.panel import Panel
from rich.table import Table


if Path("").absolute().name == "eval-agents":
    print(f"Working directory: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"Working directory set to: {Path('').absolute()}")

load_dotenv(verbose=True)
console = Console(width=100)

DATASET_NAME = "DeepSearchQA-Subset"

## 1. Run the Agent

All offline evaluators take their inputs from the `AgentResponse` object.
We run the agent once here and use the result across all subsequent sections.

In [ ]:
dataset = DeepSearchQADataset()
example = dataset.get_by_category("Finance & Economics")[0]

console.print(
    Panel(
        f"[bold]Category:[/bold] {example.problem_category}\n"
        f"[bold]Answer Type:[/bold] {example.answer_type}\n\n"
        f"[bold cyan]Question:[/bold cyan]\n{example.problem}\n\n"
        f"[bold yellow]Ground Truth:[/bold yellow]\n{example.answer}",
        title="Evaluation Example",
        border_style="blue",
    )
)

agent = KnowledgeGroundedAgent(enable_planning=True)
response = await run_with_display(agent, example.problem)

display_response(
    console,
    response.text,
    title="Agent Answer",
    subtitle=f"Duration: {response.total_duration_ms / 1000:.1f}s  |  Tools: {len(response.tool_calls)}  |  Replan count: {response.replan_count}",
)

## 2. Replanning Rate

The simplest evaluator — purely deterministic. It reads the `replan_count` counter
accumulated by the agent during execution (incremented each time `/*REPLANNING*/` appears
in the event stream).

| Score | Meaning |
|---|---|
| `Replanning/Count` | Raw number of replannings |
| `Replanning/Flag` | 1 if count exceeds the threshold for this question type |
| `Replanning/Ratio` | replan_count / plan_steps |

In [ ]:
threshold = get_max_replan_threshold(
    answer_type=example.answer_type,
    problem_category=example.problem_category,
)
plan_steps = len(response.plan.steps) if response.plan else 1

replan_evals = evaluate_replanning_rate(
    replan_count=response.replan_count,
    plan_steps=plan_steps,
    max_replan_threshold=threshold,
)

t = Table(title="Replanning Rate")
t.add_column("Metric", style="cyan")
t.add_column("Value", style="white")
t.add_column("Threshold / Note", style="dim")
for ev in replan_evals:
    note = f"threshold={threshold}" if ev.name == "Replanning/Flag" else ""
    t.add_row(ev.name, str(ev.value), note)
console.print(t)

## 3. Plan Quality

An LLM-as-judge grader that evaluates the research plan on five dimensions:
Coverage, Decomposition, DependencyLogic, Synthesis, Overall (each 1–5).

The plan rubric is **auto-derived** from `answer_type` and `problem_category`
for DeepSearchQA items — no manual annotation needed.

In [ ]:
plan_rubric = derive_plan_rubric(
    answer_type=example.answer_type,
    problem_category=example.problem_category,
)
console.print("[dim]Auto-derived plan rubric:[/dim]", plan_rubric)

plan_descriptions = [
    step.description for step in response.plan.steps
] if response.plan else []

plan_evals = await evaluate_plan_quality(
    question=example.problem,
    plan_steps=plan_descriptions,
    plan_rubric=plan_rubric,
)

t = Table(title="Plan Quality")
t.add_column("Dimension", style="cyan")
t.add_column("Score", style="white", justify="right")
t.add_column("Comment", style="dim")
for ev in plan_evals:
    t.add_row(ev.name, str(int(ev.value)), ev.comment or "")
console.print(t)

## 4. Tool Selection & Efficiency

Evaluates whether the agent used the right tools in the right order and avoided
wasteful or redundant calls. Six dimensions: Appropriateness, SequenceLogic,
SourceQuality, CallVolume, Redundancy, Overall.

The expected `tool_pattern` is auto-derived from the problem category.

In [ ]:
tool_pattern = derive_tool_pattern(
    problem_category=example.problem_category,
    answer_type=example.answer_type,
)
console.print("[dim]Auto-derived tool pattern:[/dim]", tool_pattern)

# Show the actual tool call sequence
seq_table = Table(title=f"Tool Calls ({len(response.tool_calls)} total)")
seq_table.add_column("#", style="dim", justify="right")
seq_table.add_column("Tool", style="cyan")
seq_table.add_column("Args (preview)", style="white")
for i, tc in enumerate(response.tool_calls, 1):
    args_str = str(tc.get("args", {}))
    seq_table.add_row(str(i), tc.get("name", "?"), args_str[:60])
console.print(seq_table)

tool_evals = await evaluate_tool_selection(
    question=example.problem,
    tool_calls=response.tool_calls,
    final_answer=response.text,
    tool_pattern=tool_pattern,
)

t = Table(title="Tool Selection & Efficiency")
t.add_column("Dimension", style="cyan")
t.add_column("Score", style="white", justify="right")
t.add_column("Comment", style="dim")
for ev in tool_evals:
    t.add_row(ev.name, str(int(ev.value)), ev.comment or "")
console.print(t)

## 5. Source Validation

Evaluates whether cited sources are high-authority for the question's domain.
Enforces the five-tier hierarchy defined in the system prompt.

The `source_rubric` (expected tiers + domain examples) is **fully auto-derivable**
from `problem_category` via `CATEGORY_SOURCE_RUBRIC` — no per-item annotation needed.

> **vertex_search sources** are skipped — they are already grounded in the private KB.

In [ ]:
source_rubric = get_source_rubric(example.problem_category)
console.print("[dim]Source rubric for category:[/dim]", source_rubric)

# Sources are GroundingChunk objects on AgentResponse
sources_dict = [{"title": s.title, "uri": s.uri} for s in response.sources]
console.print(f"[dim]{len(sources_dict)} sources cited[/dim]")

source_evals = await evaluate_source_validation(
    question=example.problem,
    problem_category=example.problem_category,
    sources=sources_dict,
    source_rubric=source_rubric,
)

t = Table(title="Source Validation")
t.add_column("Dimension", style="cyan")
t.add_column("Score", style="white", justify="right")
t.add_column("Comment", style="dim")
for ev in source_evals:
    t.add_row(ev.name, str(int(ev.value)), ev.comment or "")
console.print(t)

## 6. Full Experiment: All Evaluators via `run_experiment`

Wire every evaluator together and run the full offline experiment against
the Langfuse dataset. `KnowledgeQATask` returns a rich dict that all evaluators
read from — no multiple agent runs.

> This will make one agent call + six evaluator calls per dataset item.
> With 10 items and `max_concurrency=2`, expect ~20–40 minutes.

In [ ]:
task = KnowledgeQATask()

experiment_result = run_experiment(
    DATASET_NAME,
    name="knowledge-agent-full-offline-eval",
    description="All Phase 2 offline evaluators: F1, replanning, plan quality, tool selection, source validation",
    task=task.run,
    evaluators=[
        deepsearchqa_evaluator,
        replanning_evaluator,
        plan_quality_evaluator,
        tool_selection_evaluator,
        knowledge_base_evaluator,
        source_validation_evaluator,
    ],
    max_concurrency=2,
)

console.print("[green]Experiment complete[/green]")
if hasattr(experiment_result, "dataset_run_url") and experiment_result.dataset_run_url:
    display(HTML(
        f'<p>View in Langfuse: <a href="{experiment_result.dataset_run_url}" target="_blank">'
        f'{experiment_result.dataset_run_url}</a></p>'
    ))

## 7. Inspecting Results

All scores are visible in Langfuse's experiment UI. We can also inspect
item-level results programmatically for quick analysis.

In [ ]:
rows = []
for item_result in experiment_result.item_results:
    item = item_result.item
    question = str(item.input)
    row = {"question": question[:50] + "..." if len(question) > 50 else question}
    for ev in item_result.evaluations or []:
        row[ev.name] = ev.value
    rows.append(row)

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Aggregate means for key metrics
numeric_metrics = [c for c in df.columns if c != "question"]
if numeric_metrics:
    means = Table(title="Mean Scores Across Dataset")
    means.add_column("Metric", style="cyan")
    means.add_column("Mean", style="white", justify="right")
    for col in sorted(numeric_metrics):
        if df[col].dtype in ("float64", "int64"):
            means.add_row(col, f"{df[col].mean():.3f}")
    console.print(means)

## Summary

In this notebook you:

1. **Ran** the agent on a sample question and captured the full `AgentResponse`
2. **Evaluated replanning rate** deterministically from accumulated counters
3. **Evaluated plan quality** using an LLM judge on the research plan structure
4. **Evaluated tool selection** for appropriateness, sequence logic, and efficiency
5. **Evaluated source quality** against the authority tier hierarchy
6. **Ran the full experiment** with all six evaluators wired into `run_experiment`
7. **Inspected results** both in the Langfuse UI and programmatically

To iterate, change the agent configuration and re-run with a new `name` argument.
Langfuse will create a new experiment run and let you compare side-by-side.